In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib
import matplotlib.pyplot as plt
import netCDF4 as nc4

In [ ]:
matplotlib.rcParams['figure.figsize'] = [6.0, 3.375]

In [ ]:
EULER_FILE_TEMPLATE = '../SPAECIES-bld/rainshaft_{}.nc'
REF_TIME_STEP_MS = 1
EULER_TIME_STEPS_MS = [2**i * 125 for i in range(7)]
ARK2_FILE_NAME = '../SPAECIES-bld/rainshaft_ark2.nc'
ARK3_FILE_NAME = '../SPAECIES-bld/rainshaft_ark3.nc'
ARK4_FILE_NAME = '../SPAECIES-bld/rainshaft_ark4.nc'

In [ ]:
num_euler = len(EULER_TIME_STEPS_MS)
time_steps = np.array(EULER_TIME_STEPS_MS) / 1000.

In [ ]:
def time_step_ms_to_string(time_step_ms):
    """Convert time step in ms to the string used in file names."""
    if time_step_ms % 1000 == 0:
        time_step_s = time_step_ms // 1000
        return str(time_step_s) + 's'
    else:
        return str(time_step_ms) + 'ms'

In [ ]:
ref_file_name = EULER_FILE_TEMPLATE.format(
    time_step_ms_to_string(REF_TIME_STEP_MS))
euler_file_names = {
    i: EULER_FILE_TEMPLATE.format(time_step_ms_to_string(i))
    for i in EULER_TIME_STEPS_MS
}

In [ ]:
with nc4.Dataset(ref_file_name) as data:
    z_int = data['z_int'][:]
    z_mid = 0.5 * (z_int[1:] + z_int[:-1])
    z_del = z_int[:-1] - z_int[1:]
    nr_ref = data['nr'][-1,:]
    qr_ref = data['qr'][-1,:]
    num_rhs_ref = int(data['num_rhs_evals'][:])

In [ ]:
nz = len(z_mid)

In [ ]:
def calc_error(value, ref, z_del, n):
    "Calculate average error of value in representing ref using L^n norm."
    integrand = abs(value - ref)**n
    integral = sum(integrand * z_del) / sum(z_del)
    return integral**(1./n)

In [ ]:
euler_qrs = {}
euler_num_rhs = {}
euler_qr_errors = {}
for step in EULER_TIME_STEPS_MS:
    with nc4.Dataset(euler_file_names[step]) as data:
        euler_qrs[step] = data['qr'][-1,:]
        euler_num_rhs[step] = int(data['num_rhs_evals'][:])
    euler_qr_errors[step] = calc_error(euler_qrs[step], qr_ref, z_del, 2)

In [ ]:
with nc4.Dataset(ARK2_FILE_NAME) as data:
    ark2_qr = data['qr'][-1,:]
    ark2_num_rhs = int(data['num_rhs_evals'][:])
ark2_qr_error = calc_error(ark2_qr, qr_ref, z_del, 2)

In [ ]:
with nc4.Dataset(ARK3_FILE_NAME) as data:
    ark3_qr = data['qr'][-1,:]
    ark3_num_rhs = int(data['num_rhs_evals'][:])
ark3_qr_error = calc_error(ark3_qr, qr_ref, z_del, 2)

In [ ]:
with nc4.Dataset(ARK4_FILE_NAME) as ark4_data:
    ark4_qr = data['qr'][-1,:]
    ark4_num_rhs = int(data['num_rhs_evals'][:])
ark4_qr_error = calc_error(ark4_qr, qr_ref, z_del, 2)

In [ ]:
kg_to_g = 1.e3
plt.plot(qr_ref * kg_to_g, z_mid, 'k', label='Reference (Euler 10 ms)')
plt.plot(euler_qrs[8000] * kg_to_g, z_mid, '--', color='orange', label='Euler 8s')
plt.plot(euler_qrs[4000] * kg_to_g, z_mid, '-', color='orange', label='Euler 4s')
plt.plot(ark2_qr * kg_to_g, z_mid, 'r', label='ARKODE 2nd-order')
plt.plot(ark3_qr * kg_to_g, z_mid, 'g', label='ARKODE 3rd-order')
plt.plot(ark4_qr * kg_to_g, z_mid, 'b', label='ARKODE 4th-order')
plt.xlabel('Rain mass mixing ratio (g/kg)')
plt.xlim(0., 0.1)
plt.ylabel('Altitude (m)')
plt.legend()
plt.savefig('qr_profile.png', bbox_inches='tight')

In [ ]:
euler_num_rhs_plot = np.array([euler_num_rhs[step] for step in EULER_TIME_STEPS_MS])
euler_qr_errors_plot = np.array([euler_qr_errors[step] for step in EULER_TIME_STEPS_MS])

In [ ]:
plt.loglog((euler_num_rhs_plot[1] * 2, euler_num_rhs_plot[1] / 32),
           (euler_qr_errors_plot[1] * kg_to_g / 2, euler_qr_errors_plot[1] * kg_to_g * 32),
           'k--', label='Expected 1st-order convergence')
plt.loglog(euler_num_rhs_plot, euler_qr_errors_plot * kg_to_g, 'o-', color='orange', label='Forward Euler')
plt.loglog(ark2_num_rhs, ark2_qr_error * kg_to_g, 'ro', label='ARKODE default 2nd-order')
plt.loglog(ark3_num_rhs, ark3_qr_error * kg_to_g, 'bo', label='ARKODE default 3rd-order')
plt.loglog(ark4_num_rhs, ark4_qr_error * kg_to_g, 'go', label='ARKODE default 4th-order')
plt.xlabel('Total # of function evaluations')
plt.ylabel('RMS error in $q_r$ (g/kg)')
plt.legend()
plt.savefig('qr_convergence.png', bbox_inches='tight')